# Tutorial: Document Classification Using CNN

In this tutorial, we will learn how to build CNN model in keras and tensorflow for document classification.

Steps:
 - Load the dataset (topic categorization dataset)
 - Clean the dataset
 - Apply Tokenizer 
 - Sequence padding
 - Build the CNN model
 - Use Label Encoder
 - Train the model
 - Evaluate the model
 - Use the model on new data
 

## Load Dataset

The dataset is called “Twenty Newsgroups”. It is a collection of approximately 20,000 newsgroup documents, partitioned evenly across 20 different newsgroups.
In order to get faster, we will work on a partial dataset with only 4 categories out of the 20 available in the dataset.

In [1]:
from sklearn.datasets import fetch_20newsgroups

categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

twenty_train = fetch_20newsgroups(subset='train',
    categories=categories, shuffle=True, random_state=42)

twenty_test = fetch_20newsgroups(subset='test',
    categories=categories, shuffle=True, random_state=42)

## Cleaning the data before modeling

In [2]:
import re

def text_cleaning(text):
    text = text.lower() # convert to lower case
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' ', text) # remove emails
    text = re.sub(r'subject:.*',' ',text) # remove lines start with Subject
    text = re.sub(r'write to:.*',' ',text) # remove lines start with Write to:
    text = re.sub(r'from:.*',' ',text) # remove lines start with From:
    
    text = re.sub(r'nntp-posting-host:.*',' ',text) # remove lines start with Nntp-Posting-Host:
    text = re.sub(r'lines:.*',' ',text) # remove lines start with Lines:
    text = re.sub(r'organization:.*',' ',text) # remove lines start with Organization:
    text = re.sub(r'email:.*',' ',text) # remove lines start with email:
    text = re.sub(r'tel:.*',' ',text) # remove lines start with tel:
    text = re.sub(r'fax:.*',' ',text) # remove lines start with fax:
    
    text = re.sub("\(.*?\)","",text) # Delete all the data which are present in the brackets.
    
    # Decontractions, replace words like below to full words.
    text = re.sub(r"can\'t","can not",text)
    text = re.sub(r"\'s","is",text)
    text = re.sub(r"i\'ve'","i have",text)
    text = re.sub(r"i\'m","i am",text)
    text = re.sub(r"you\'re","you are",text)
    text = re.sub(r"i\'ll","i will",text)
    
    text = re.sub(r'[^A-Za-z_]'," ",text) # Replace all the words except "A-Za-z_" with space. 
    text = re.sub(r"[0-9]","",text) # Replace all the digits with space i.e delete all the digits. 
    
    return text


for i, text in enumerate(twenty_train.data):
    twenty_train.data[i] = text_cleaning(text)
    
for i, text in enumerate(twenty_test.data):
    twenty_test.data[i] = text_cleaning(text)



## Apply Tokenizer
It converts text into sequences of integers, where each integer represents a token in the text. 

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

top_words = 7000

# adding a tokenizer 
tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n', num_words=top_words)
tokenizer.fit_on_texts(twenty_train.data)
x_train  = tokenizer.texts_to_sequences(twenty_train.data)
x_test = tokenizer.texts_to_sequences(twenty_test.data)

# print(tokenizer.word_index)


## Apply Sequence Padding

In [5]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 450

x_train = pad_sequences(x_train, maxlen=max_words, padding="post")
x_test= pad_sequences(x_test, maxlen=max_words, padding='post')

## Build the CNN model

In [6]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.layers.embeddings import Embedding
from keras.preprocessing import sequence

# Building the CNN Model
model = Sequential()      # initilaizing the Sequential nature for CNN model
# Adding the embedding layer which will take in maximum of 450 words as input and provide a 32 dimensional output of those words which belong in the top_words dictionary
model.add(Embedding(top_words, 32, input_length=max_words))
model.add(Conv1D(32, 3, padding='same', activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(250, activation='relu'))
model.add(Dense(4, activation='softmax'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 450, 32)           224000    
_________________________________________________________________
conv1d (Conv1D)              (None, 450, 32)           3104      
_________________________________________________________________
max_pooling1d (MaxPooling1D) (None, 225, 32)           0         
_________________________________________________________________
flatten (Flatten)            (None, 7200)              0         
_________________________________________________________________
dense (Dense)                (None, 250)               1800250   
_________________________________________________________________
dense_1 (Dense)              (None, 4)                 1004      
Total params: 2,028,358
Trainable params: 2,028,358
Non-trainable params: 0
______________________________________________

## Label Encoder

In [7]:
from sklearn.preprocessing import LabelEncoder
from keras.utils import np_utils
## encoding lables  Encode target labels with value between 0 and n_classes-1.
encoder = LabelEncoder()
encoder.fit(twenty_train.target)
y_encoder = encoder.transform(twenty_train.target)
y_train = np_utils.to_categorical(y_encoder)

encoder = LabelEncoder()
encoder.fit(twenty_test.target)
y_encoder = encoder.transform(twenty_test.target)
y_test = np_utils.to_categorical(y_encoder)

## Train the model

In [8]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=20, batch_size=128, verbose=2)

Epoch 1/20
18/18 - 1s - loss: 0.5610 - accuracy: 0.3381 - val_loss: 0.5410 - val_accuracy: 0.3742
Epoch 2/20
18/18 - 1s - loss: 0.5042 - accuracy: 0.4440 - val_loss: 0.4981 - val_accuracy: 0.4268
Epoch 3/20
18/18 - 1s - loss: 0.4145 - accuracy: 0.6460 - val_loss: 0.4069 - val_accuracy: 0.5899
Epoch 4/20
18/18 - 1s - loss: 0.2915 - accuracy: 0.8028 - val_loss: 0.3406 - val_accuracy: 0.7217
Epoch 5/20
18/18 - 1s - loss: 0.1809 - accuracy: 0.9207 - val_loss: 0.3010 - val_accuracy: 0.7230
Epoch 6/20
18/18 - 1s - loss: 0.0985 - accuracy: 0.9663 - val_loss: 0.2655 - val_accuracy: 0.7929
Epoch 7/20
18/18 - 1s - loss: 0.0483 - accuracy: 0.9898 - val_loss: 0.2451 - val_accuracy: 0.8123
Epoch 8/20
18/18 - 1s - loss: 0.0253 - accuracy: 0.9942 - val_loss: 0.2572 - val_accuracy: 0.8242
Epoch 9/20
18/18 - 1s - loss: 0.0142 - accuracy: 0.9987 - val_loss: 0.2482 - val_accuracy: 0.8342
Epoch 10/20
18/18 - 1s - loss: 0.0091 - accuracy: 0.9996 - val_loss: 0.2582 - val_accuracy: 0.8282
Epoch 11/20
18/18 -

## Evaluate the model on the testing data

In [9]:
# Getting score metrics from our model
scores = model.evaluate(x_test, y_test, verbose=0)
# Displays the accuracy of correct sentiment prediction over test data
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 83.69%


## Use the model on new data

In [10]:
docs_new = [
    """A graphics processing unit (GPU) is a specialized electronic circuit initially designed to accelerate computer graphics and image processing (either on a video card or embedded on motherboards, mobile phones, personal computers, workstations, and game consoles).""", 
    """Paracetamol (acetaminophen[a] or para-hydroxyacetanilide) is a non-opioid analgesic and antipyretic agent used to treat fever and mild to moderate pain. It is a widely used over the counter medication""",
    """A wide range of beliefs and practices are found across the world among those who call themselves Christian. Denominations and sects disagree on a common definition of "Christianity". For example, Timothy Beal notes the disparity of beliefs among those who identify as Christians in the United States."""
]

for i, doc in enumerate(docs_new):
    docs_new[i] = text_cleaning(doc)

doc_seq = tokenizer.texts_to_sequences(docs_new)
doc_seq = pad_sequences(doc_seq, maxlen=max_words, padding="post")
predict = model.predict(doc_seq)

import numpy as np
print(twenty_test.target_names)
print(np.round(predict))



['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']
[[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
